# 🧭 Chapter 09-02a: MCP — Model Context Protocol

---

## 📖 The Story: The Universal Adapter

Remember when every phone had a different charger? USB, Micro-USB, Lightning, proprietary pins... Then USB-C came along as *one standard to rule them all*.

**MCP** is the USB-C of AI tool integration. Instead of every AI app building custom integrations for every tool, MCP provides ONE standard protocol.

---

## 🎯 What is MCP?

```
╔══════════════════════════════════════════════════════════════════════╗
║  MODEL CONTEXT PROTOCOL (MCP)                                       ║
║                                                                      ║
║  An open standard (by Anthropic) that defines how AI models        ║
║  connect to external tools, data sources, and services.            ║
║                                                                      ║
║  Before MCP: N models × M tools = N×M custom integrations         ║
║  With MCP:   N models + M tools = N+M implementations              ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 🏗️ MCP Architecture

```
   ┌─────────────┐     ┌────────────────┐     ┌──────────────┐
   │  MCP Host    │     │  MCP Client     │     │  MCP Server  │
   │  (Claude,    │────►│  (Protocol      │────►│  (Your tool) │
   │   VS Code,   │     │   handler)      │     │              │
   │   Your app)  │     │                 │     │  - Tools     │
   └─────────────┘     └────────────────┘     │  - Resources │
                                               │  - Prompts   │
                                               └──────────────┘
```

### Three Components:
1. **Host** — The AI application (Claude Desktop, VS Code, custom app)
2. **Client** — Protocol handler inside the host
3. **Server** — Your tool/service that exposes capabilities

### Three Primitives:
| Primitive | Direction | Description |
|-----------|-----------|-------------|
| **Tools** | Server → Model | Functions the model can call |
| **Resources** | Server → Model | Data the model can read (like files) |
| **Prompts** | Server → Model | Pre-built prompt templates |

---

## 🔌 Transport Mechanisms

How MCP clients talk to MCP servers:

| Transport | Best For | How |
|-----------|----------|-----|
| **stdio** | Local tools | Server runs as subprocess |
| **SSE** | Remote services | HTTP + Server-Sent Events |
| **Streamable HTTP** | Production | HTTP with streaming |

```python
# stdio: Host spawns server as child process
# Communication via stdin/stdout (JSON-RPC)

# SSE: Server runs on HTTP endpoint
# Client connects via Server-Sent Events
```

---

## 🛠️ Building an MCP Server

```python
# Example: Simple MCP server with Python SDK
from mcp.server import Server
from mcp.types import Tool, TextContent

server = Server("my-tools")

@server.list_tools()
async def list_tools():
    return [
        Tool(
            name="get_weather",
            description="Get weather for a city",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {"type": "string"}
                },
                "required": ["city"]
            }
        )
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "get_weather":
        city = arguments["city"]
        return [TextContent(type="text", text=f"72°F in {city}")]

# Run with: python server.py (stdio transport)
```

---

## 📦 Popular MCP Servers

| Server | Purpose |
|--------|---------|
| `@modelcontextprotocol/server-filesystem` | File system access |
| `@modelcontextprotocol/server-github` | GitHub API |
| `@modelcontextprotocol/server-postgres` | Database queries |
| `@modelcontextprotocol/server-slack` | Slack messaging |
| `@modelcontextprotocol/server-brave-search` | Web search |

### Claude Desktop Configuration
```json
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/path/to/allowed/dir"
      ]
    }
  }
}
```

---

## 🎓 Interview Corner

**Q: What is MCP and how does it compare to function calling?**

<details>
<summary>💡 Click for Expert Answer</summary>

**Function calling** is a model-level feature — the LLM outputs structured tool invocations.

**MCP** is an application-level protocol — it standardizes how apps DISCOVER and CONNECT to tools.

They're complementary:
1. MCP server exposes tools with schemas
2. MCP client fetches available tools
3. Client sends tool schemas to LLM (function calling)
4. LLM requests tool calls
5. Client executes via MCP protocol
6. Results sent back to LLM

MCP solves the N×M integration problem — connect ANY model to ANY tool with ONE protocol.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 MCP = open standard for AI ↔ tool integration (the USB-C).    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Architecture: Host → Client → Server. Three primitives.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Transports: stdio (local), SSE (remote), HTTP (production).    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 MCP + function calling = complete tool integration stack.      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Growing ecosystem: filesystem, GitHub, Postgres, Slack, etc.   │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On — Build Your Own MCP Server!

---

### 🎉 MCP Protocol Understood!
The universal adapter for AI tools — one protocol, infinite possibilities. 🚀